# 169 — Residual Stream Probing

Probe the residual stream for specific information: token identity recovery,
next-token predictions, directional tracking, prediction quality,
and information content.

## Why This Matters

Residual stream stream probing characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_probing import (
    token_identity_probe,
    next_token_probe,
    directional_probe,
    layer_prediction_quality,
    residual_information_content,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
for l in range(4):
    for p in range(0, 15, 3):
        result = token_identity_probe(model, tokens, layer=l, pos=p)
        print(f"L{l} pos{p}: true={result['true_token']}  pred={result['predicted_token']}  "
              f"rank={result['true_token_rank']}  sim={result['true_token_similarity']:.3f}")
    print()

In [ ]:
for l in range(4):
    result = next_token_probe(model, tokens, layer=l)
    print(f"Layer {l}: pred={result['top_prediction']}  logit={result['top_logit']:.3f}  "
          f"prob={result['top_probability']:.4f}  H={result['entropy']:.3f}")

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(7), (32,))
result = directional_probe(model, tokens, direction, label='random_dir')
print(f"Tracking: {result['direction_label']}\n")
for p in result['per_layer']:
    bar = '#' * int(abs(p['mean_projection']) * 20)
    print(f"Layer {p['layer']}: mean={p['mean_projection']:+.4f}  "
          f"std={p['std']:.4f}  range=[{p['min_projection']:.3f}, {p['max_projection']:.3f}]")

In [ ]:
result = layer_prediction_quality(model, tokens)
print(f"Final prediction: token {result['final_prediction']}\n")
for p in result['per_layer']:
    match = 'MATCH' if p['matches_final'] else f'rank {p["final_token_rank"]}'
    print(f"Layer {p['layer']}: pred={p['top_prediction']}  {match}  "
          f"logit={p['final_token_logit']:.3f}  H={p['entropy']:.3f}")

In [ ]:
result = residual_information_content(model, tokens)
for p in result['per_layer']:
    print(f"Layer {p['layer']}: norm={p['norm']:.3f}  eff_dim={p['effective_dimensionality']:.1f}  "
          f"sparsity={p['sparsity']:.3f}")